# 03 – Repeat-Purchase Propensity Model (out-of-time validation)
Question: *Which customers will buy again in the next 6 months?*  Use: target retention campaigns.

**Leakage control:** features only use orders BEFORE a cutoff date; the label only looks AFTER it.
Train on an earlier cutoff, test on a later one (out-of-time), which is how it would be used in real life.

## 0. Setup & config
**Edit only this cell** to match your tables. Defaults follow the usual AdventureWorks CSV schema
(Sales_2015/16/17, Customers, Products, Territories). If you read from your Gold layer in Databricks,
set `USE_SPARK = True` and fix `GOLD_PATH` and the table names.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None); pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

USE_SPARK  = False                 # True inside Databricks (reads Delta from the Gold layer)
DATA_DIR   = "data"                # local folder with CSV exports (used when USE_SPARK = False)
GOLD_PATH  = "/mnt/gold"           # Databricks mount point of your gold container

SALES_TABLES    = ["Sales_2015", "Sales_2016", "Sales_2017"]   # or ["fact_sales"] if you have one Gold fact table
CUSTOMERS_TABLE = "Customers"
PRODUCTS_TABLE  = "Products"
TERRITORY_TABLE = "Territories"    # optional; set to None to skip

C = dict(order_date="OrderDate", order_no="OrderNumber", product="ProductKey", customer="CustomerKey",
         territory="TerritoryKey", qty="OrderQuantity", price="ProductPrice", cost="ProductCost")

for d in ("figures", "outputs", "audit"):
    os.makedirs(d, exist_ok=True)

def load(name):
    if USE_SPARK:
        return spark.read.format("delta").load(f"{GOLD_PATH}/{name}").toPandas()   # noqa: F821 (spark exists in Databricks)
    return pd.read_csv(f"{DATA_DIR}/{name}.csv")

## 1. Load tables and build one clean order-line table

In [ ]:
sales     = pd.concat([load(t) for t in SALES_TABLES], ignore_index=True)
customers = load(CUSTOMERS_TABLE)
products  = load(PRODUCTS_TABLE)
raw_tables = {"sales": sales, "customers": customers, "products": products}

## 2. Clean order lines

In [ ]:
df = sales.drop_duplicates().dropna(subset=[C["customer"], C["order_date"]]).copy()
df[C["order_date"]] = pd.to_datetime(df[C["order_date"]])
df[C["customer"]] = df[C["customer"]].astype(int)
df = df[df[C["qty"]] > 0].merge(products[[C["product"], C["price"]]], on=C["product"], how="left")
df["revenue"] = df[C["qty"]] * df[C["price"]]
lines = df.rename(columns={C["customer"]: "cust", C["order_no"]: "order", C["order_date"]: "date", C["product"]: "prod", C["qty"]: "qty"})
print(lines.date.min().date(), "->", lines.date.max().date())

## 3. Build features (history before cutoff) and label (purchase in the next H days)

In [ ]:
H = 180                                            # prediction horizon in days
max_date = lines.date.max()
CUT_TRAIN = max_date - pd.Timedelta(days=2 * H)    # train: features before this, label in the following H days
CUT_TEST  = max_date - pd.Timedelta(days=H)        # test : later period the model has never seen
print("train cutoff:", CUT_TRAIN.date(), "| test cutoff:", CUT_TEST.date(), "| last data:", max_date.date())

def build(cutoff):
    hist = lines[lines.date < cutoff]
    fut  = lines[(lines.date >= cutoff) & (lines.date < cutoff + pd.Timedelta(days=H))]
    orders = hist.groupby(["cust", "order"], as_index=False).agg(date=("date", "max"), revenue=("revenue", "sum"), items=("qty", "sum"))
    g = orders.groupby("cust")
    X = pd.DataFrame({
        "recency_days":   (cutoff - g.date.max()).dt.days,
        "frequency":      g.order.nunique(),
        "monetary":       g.revenue.sum(),
        "avg_order_value": g.revenue.mean(),
        "avg_items":      g["items"].mean(),
        "tenure_days":    (cutoff - g.date.min()).dt.days,
    })
    X["distinct_products"] = hist.groupby("cust")["prod"].nunique()
    X["avg_gap_days"] = (X.tenure_days - X.recency_days) / (X.frequency - 1).replace(0, np.nan)
    X["avg_gap_days"] = X["avg_gap_days"].fillna(X.tenure_days)              # single-order customers: use tenure
    y = X.index.isin(fut.cust.unique()).astype(int)
    return X, pd.Series(y, index=X.index, name="bought_again")

X_train, y_train = build(CUT_TRAIN)
X_test,  y_test  = build(CUT_TEST)
print(f"train: {X_train.shape}, repeat rate {y_train.mean():.1%} | test: {X_test.shape}, repeat rate {y_test.mean():.1%}")

## 4. Quick EDA of features vs target

In [ ]:
eda = X_train.assign(bought_again=y_train)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, col in zip(ax, ["recency_days", "frequency", "monetary"]):
    sns.boxplot(data=eda, x="bought_again", y=col, ax=a, showfliers=False); a.set_title(f"{col} vs repeat purchase")
plt.tight_layout(); plt.savefig("figures/08_features_vs_target.png", dpi=150); plt.show()

## 5. Models: baseline vs Logistic Regression vs Random Forest

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

models = {
    "Baseline (always majority)": DummyClassifier(strategy="prior"),
    "Logistic Regression":        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest":              RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
}
rows, fitted = [], {}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
for name, m in models.items():
    cv_auc = cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc").mean()      # 5-fold CV on train period
    m.fit(X_train, y_train); fitted[name] = m
    p = m.predict_proba(X_test)[:, 1]; pred = m.predict(X_test)                          # out-of-time test
    rows.append({"model": name, "CV AUC (train)": cv_auc, "Test ROC-AUC": roc_auc_score(y_test, p),
                 "Test PR-AUC": average_precision_score(y_test, p), "Precision": precision_score(y_test, pred, zero_division=0),
                 "Recall": recall_score(y_test, pred), "F1": f1_score(y_test, pred)})
results = pd.DataFrame(rows).set_index("model").round(3)
results.to_csv("outputs/model_results.csv"); results

Why these metrics: accuracy is misleading when classes are imbalanced. **ROC-AUC** = how well the model ranks
repeat buyers above non-buyers; **PR-AUC / precision / recall** show the trade-off when you can only contact a few customers.

## 6. Feature importance and business lift

In [ ]:
from sklearn.inspection import permutation_importance
best_name = results["Test ROC-AUC"].idxmax(); best = fitted[best_name]; print("Best model:", best_name)
pi = permutation_importance(best, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=42)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values()
imp.plot(kind="barh", figsize=(6, 4), title=f"Permutation importance ({best_name})"); plt.tight_layout()
plt.savefig("figures/09_feature_importance.png", dpi=150); plt.show()

In [ ]:
# Lift: if we contact only the top 20% highest-scored customers, how many repeat buyers do we reach?
score = pd.Series(best.predict_proba(X_test)[:, 1], index=X_test.index)
tmp = pd.DataFrame({"score": score, "y": y_test}).sort_values("score", ascending=False)
top = tmp.head(int(len(tmp) * 0.2))
capture = top.y.sum() / tmp.y.sum()
print(f"Top 20% by score captures {capture:.1%} of all repeat buyers (random targeting would capture 20%) -> lift {capture/0.2:.2f}x")

In [ ]:
# Save scores (write back to Gold in Databricks the same way as notebook 02)
out = tmp.reset_index().rename(columns={"cust": "CustomerKey", "score": "repeat_purchase_probability", "y": "actual_bought_again"})
out.to_csv("outputs/customer_repeat_scores.csv", index=False)
out.head()

## 7. Limitations (say these in the interview – it shows maturity)
- AdventureWorks is a practice dataset, so the numbers are illustrative, not real business results.
- Only purchase-behaviour features are used; real models would add demographics, channel and product-category features.
- Threshold 0.5 is a default; in practice choose it from campaign cost vs benefit.